# 00 Ingest GUI Workflow (`adamacs_ingest_v2`)

Use the GUI-first workflow to configure and run ingestion jobs safely.


In [1]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


## 1) Setup and DB connection

Set `DJ_HOST` and `DJ_USER` in your shell if needed. Password is prompted by the DataJoint connection flow.


In [2]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')

repo_root = Path.cwd()
config_path = repo_root / 'dj_local_conf.json'

import datajoint as dj
if config_path.exists():
    dj.config.load(str(config_path))
else:
    print(f'Warning: config not found at {config_path}; relying on environment variables.')

print(f'Working directory: {repo_root}')
print(f'DataJoint version: {dj.__version__}')
print(f"DB prefix: {dj.config.get('custom', {}).get('database.prefix', '<unset>')}")
dj.conn()


Working directory: /home/backup_user/adamacs_ingest
DataJoint version: 0.14.8
DB prefix: roselab_


DataJoint connection (connected) tobiasr@172.25.64.3:3306

## 2) Load ingest v2 helper

In [3]:
import adamacs.helpers.adamacs_ingest_v2 as ai
print('Loaded:', ai.__name__)


RSpace client not found or failed to initialize: module 'rspace_client' has no attribute 'Client'. RSpace functionality will be disabled.
Loaded: adamacs.helpers.adamacs_ingest_v2


## 3) Discover session folders

In [ ]:
# Notebook parameters (edit here for local runs).
# ADAMACS_DATE_FILTER examples:
#   *                      -> no date filtering
#   2025-01-20             -> exact date
#   >=2025-01-01           -> on/after date
#   <2025-02-01            -> before date
#   2025-01-01:2025-01-31  -> inclusive date range
#   >2025-05-01*           -> comparator with optional wildcard suffix
#   2025-01                -> substring fallback
ADAMACS_SESSION_FILTER = "*NK*"
ADAMACS_DATE_FILTER = ">2025-06-01*"
ADAMACS_LAUNCH_GUI = True

print("ADAMACS_SESSION_FILTER =", ADAMACS_SESSION_FILTER)
print("ADAMACS_DATE_FILTER    =", ADAMACS_DATE_FILTER)
print("ADAMACS_LAUNCH_GUI     =", ADAMACS_LAUNCH_GUI)


ADAMACS_SESSION_FILTER = *NK*
ADAMACS_DATE_FILTER    = >2025-06-01*
ADAMACS_LAUNCH_GUI     = False


In [9]:
import os
from natsort import natsorted
from IPython.display import display, HTML

session_filter = ADAMACS_SESSION_FILTER
date_filter = ADAMACS_DATE_FILTER

root_dirs = dj.config.get("custom", {}).get("exp_root_data_dir", [])
if not root_dirs:
    raise ValueError('dj.config["custom"]["exp_root_data_dir"] is not configured.')

dataroot = root_dirs[0]
all_session_dirs = [
    d for d in os.listdir(dataroot) if os.path.isdir(os.path.join(dataroot, d))
]

dirs_root, parsed_date_filter, unmatched_date_dirs = ai.filter_session_dirs(
    all_session_dirs,
    session_filter=session_filter,
    date_filter=date_filter,
)
sorted_dirs_root = natsorted(dirs_root, reverse=True)

print(f"Data root: {dataroot}")
print(f"Filter: session={session_filter!r}, date={date_filter!r} ({parsed_date_filter['mode']})")
print(f"Found {len(sorted_dirs_root)} candidate sessions.")

if parsed_date_filter["mode"] != "substring" and unmatched_date_dirs:
    print(
        f"Note: {len(unmatched_date_dirs)} session folders had no parseable date and were skipped by date comparison."
    )

for path in sorted_dirs_root:
    display(HTML(f'<a href="{os.path.join(dataroot, path)}" target="_blank">{path}</a>'))


Data root: /datajoint-data/data/nataliak
Filter: session='*NK*', date='>2025-06-01*' (cmp)
Found 3 candidate sessions.
Note: 21 session folders had no parseable date and were skipped by date comparison.


## 4) Launch ingest GUI (opt-in)

Set `ADAMACS_LAUNCH_GUI = True` in the parameter cell before running this cell to open the interactive GUI.


In [10]:
if not sorted_dirs_root:
    print('No matching sessions found. Adjust ADAMACS_SESSION_FILTER / ADAMACS_DATE_FILTER and rerun.')
elif not ADAMACS_LAUNCH_GUI:
    print('GUI launch skipped. Set ADAMACS_LAUNCH_GUI = True in the parameter cell to open the interactive selector.')
    print('First 10 matching sessions:')
    for path in sorted_dirs_root[:10]:
        print('  -', path)
else:
    print('ADAMACS INGEST GUI v2')
    selected_data, get_dlc_models = ai.select_sessions(
        sorted_dirs_root,
        do_population=False,
        rspace_upload=False,
        ingest_opt='trigger',
    )


GUI launch skipped. Set ADAMACS_LAUNCH_GUI = True in the parameter cell to open the interactive selector.
First 10 matching sessions:
  - NK_ROS-9999_2025-07-07_scan9FV17VYO_sess9FV17VYO
  - NK_ROS-9999_2025-07-07_scan9FV17SNO_sess9FV17SNO
  - NK_ROS-9999_2025-07-07_scan9FV17P1I_sess9FV17P1I


## Notes

- Use this notebook as the default ingest entrypoint.
- Keep ad-hoc debugging in separate notebooks/scripts.
